In [1]:
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import RobertaTokenizerFast, RobertaModel, TrainingArguments, Trainer
from sklearn.metrics import f1_score



In [2]:


#Load the dataset directly from a URL (if hosted as CSV or JSON)// we get the data in json form only 

train_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")
print(train_data[1])  # Print the first example to understand its structure

train_df = train_data.to_pandas()
print("Sample data (first 5 rows):")
print(train_df.head())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

eng/train-00000-of-00001.parquet:   0%|          | 0.00/179k [00:00<?, ?B/s]

eng/dev-00000-of-00001.parquet:   0%|          | 0.00/11.8k [00:00<?, ?B/s]

eng/test-00000-of-00001.parquet:   0%|          | 0.00/182k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2763 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/115 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2765 [00:00<?, ? examples/s]

{'id': 'eng_train_track_b_00002', 'text': 'This involved swimming a pretty large lake that was over my head.', 'anger': 0, 'disgust': None, 'fear': 2, 'joy': 0, 'sadness': 0, 'surprise': 0}
Sample data (first 5 rows):
                        id                                               text  \
0  eng_train_track_b_00001                       Colorado, middle of nowhere.   
1  eng_train_track_b_00002  This involved swimming a pretty large lake tha...   
2  eng_train_track_b_00003        It was one of my most shameful experiences.   
3  eng_train_track_b_00004  After all, I had vegetables coming out my ears...   
4  eng_train_track_b_00005                        Then the screaming started.   

   anger  disgust  fear  joy  sadness  surprise  
0      0      NaN     1    0        0         1  
1      0      NaN     2    0        0         0  
2      0      NaN     1    0        3         0  
3      0      NaN     0    0        0         0  
4      0      NaN     3    0        1        

In [3]:
EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def fill_missing_intensities(example):
    for e in EMOTIONS:
        if example[e] is None:      # replace the None of digust with 0
            example[e] = 0
    return example

train_data = train_data.map(fill_missing_intensities)
val_data   = val_data.map(fill_missing_intensities)
test_data  = test_data.map(fill_missing_intensities)




Map:   0%|          | 0/2763 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/2765 [00:00<?, ? examples/s]

In [7]:
def add_labels(example):
    intens = [int(example[e]) for e in EMOTIONS]                     # [0..3] x 6
    present = [1 if v > 0 else 0 for v in intens]                   # [0/1] x 6
    example["present_labels"] = present                             # Step 1 target
    example["intensity_labels"] = intens                              # Step 2 target
    return example

train_data = train_data.map(add_labels)
val_data   = val_data.map(add_labels)
test_data  = test_data.map(add_labels)

print(train_data[:5]["present_labels"]) 

print(train_data[:5]["intensity_labels"])

Map:   0%|          | 0/2763 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/2765 [00:00<?, ? examples/s]

[[0, 0, 1, 0, 0, 1], [0, 0, 1, 0, 0, 0], [0, 0, 1, 0, 1, 0], [0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 1, 1]]
[[0, 0, 1, 0, 0, 1], [0, 0, 2, 0, 0, 0], [0, 0, 1, 0, 3, 0], [0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 1, 2]]


In [12]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

train_tok = train_data.map(tokenize, batched=True)
val_tok   = val_data.map(tokenize, batched=True)
test_tok  = test_data.map(tokenize, batched=True)

print(train_tok[0])

Map:   0%|          | 0/2763 [00:00<?, ? examples/s]

Map:   0%|          | 0/115 [00:00<?, ? examples/s]

Map:   0%|          | 0/2765 [00:00<?, ? examples/s]

{'id': 'eng_train_track_b_00001', 'text': 'Colorado, middle of nowhere.', 'anger': 0, 'disgust': 0, 'fear': 1, 'joy': 0, 'sadness': 0, 'surprise': 1, 'present_labels': [0, 0, 1, 0, 0, 1], 'intensity_labels': [0, 0, 1, 0, 0, 1], 'input_ids': [0, 39557, 6, 1692, 9, 9261, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [16]:
def prepare_step1(ds):
    ds = ds.remove_columns([c for c in ds.column_names if c not in ["input_ids","attention_mask","present_labels"]])
    ds = ds.rename_column("present_labels", "labels")
    ds.set_format("torch")
    return ds

train_s1 = prepare_step1(train_tok)
val_s1   = prepare_step1(val_tok)
test_s1  = prepare_step1(test_tok)
print(train_s1[0])

{'labels': tensor([0, 0, 1, 0, 0, 1]), 'input_ids': tensor([    0, 39557,     6,  1692,     9,  9261,     4,     2]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1])}


In [17]:
class RobertaForMultiLabel(nn.Module):
    def __init__(self, model_name="roberta-base", num_labels=6):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)
        self.loss_fn = nn.BCEWithLogitsLoss()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]  
        logits = self.classifier(self.dropout(cls))

        loss = None
        if labels is not None:
            labels = labels.float()
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}
